# PJM Hourly Energy Consumption Benchmark

This notebook evaluates Persistence, Linear Regression, and Random Forest on the PJM East hourly load dataset. It uses a chronological split and saves processed data, figures, models, metrics, predictions, feature importance, and a complete run log.

## Expected project structure

```text
PJM/
├── data/
│   ├── raw/
│   │   └── PJME_hourly.csv
│   └── processed/
├── figures/
├── models/
├── notebook/
│   └── PJM_Benchmark.ipynb
└── results/
```


## 1 Imports and Configuration


In [ ]:
from datetime import datetime
from pathlib import Path
import platform
import sys
import time
import warnings

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("default")
BENCHMARK_START_TIME = time.perf_counter()
RUN_STARTED_AT = datetime.now().astimezone()

# Reproducible benchmark configuration
RANDOM_STATE = 42
TRAIN_FRACTION = 0.80
N_LAGS = 24
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 18
RF_MIN_SAMPLES_LEAF = 2
MODEL_COMPRESSION_LEVEL = 3
PREDICTION_PLOT_HOURS = 168
FIGURE_DPI = 300
TARGET_COLUMN = "PJME_MW"
DATETIME_COLUMN = "Datetime"
PROCESSED_FILENAME = "PJME_hourly_processed.csv"

# The notebook is intentionally path-relative and must run from PJM/notebooks/.
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name.lower() != "notebooks":
    raise RuntimeError(
        "Run this notebook from the PJM/notebooks directory. "
        f"Current working directory: {NOTEBOOK_DIR}"
    )
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "PJME_hourly.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

for directory in (PROCESSED_DIR, FIGURES_DIR, MODELS_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LOG_MESSAGES = []


def log_event(message):
    """Record and display a timestamped benchmark event."""
    timestamp = datetime.now().astimezone().isoformat(timespec="seconds")
    entry = f"[{timestamp}] {message}"
    LOG_MESSAGES.append(entry)
    print(entry)


def save_figure(filename):
    """Apply consistent layout, save a publication-quality PNG, and close."""
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()
    plt.close()


def save_model(model, destination):
    """Compress and atomically publish a model file."""
    temporary_path = destination.with_name(f"{destination.name}.tmp")
    if temporary_path.exists():
        temporary_path.unlink()
    try:
        joblib.dump(
            model,
            temporary_path,
            compress=("gzip", MODEL_COMPRESSION_LEVEL),
        )
        temporary_path.replace(destination)
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise


plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "font.size": 11,
})

log_event("Environment and configuration initialized.")
log_event(f"Python {platform.python_version()} on {platform.platform()}.")
log_event(f"Working directory: {NOTEBOOK_DIR}")


## 2 Dataset Loading


In [ ]:
if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError(
        "PJM source dataset not found. Place PJME_hourly.csv at: "
        f"{RAW_DATA_PATH}"
    )

raw_df = pd.read_csv(RAW_DATA_PATH)
ORIGINAL_ROWS = len(raw_df)
log_event(f"Dataset loaded from {RAW_DATA_PATH} ({ORIGINAL_ROWS:,} rows).")


## 3 Initial Inspection


In [ ]:
required_columns = {DATETIME_COLUMN, TARGET_COLUMN}
missing_columns = required_columns.difference(raw_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print("Shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.describe(include="all"))
print("Missing values:\n", raw_df[list(required_columns)].isna().sum())
print("Duplicate full rows:", int(raw_df.duplicated().sum()))
log_event("Initial dataset inspection completed.")


## 4 Data Preprocessing

Invalid datetime or target values cannot be used by the forecasting models and are removed explicitly. Valid duplicate timestamps are **retained** because PJM's daylight-saving transitions can produce repeated local clock times.


In [ ]:
df = raw_df[[DATETIME_COLUMN, TARGET_COLUMN]].copy()
df[DATETIME_COLUMN] = pd.to_datetime(df[DATETIME_COLUMN], errors="coerce")
df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")

invalid_datetime_count = int(df[DATETIME_COLUMN].isna().sum())
invalid_target_count = int(df[TARGET_COLUMN].isna().sum())
df = df.dropna(subset=[DATETIME_COLUMN, TARGET_COLUMN])
df = df.sort_values(DATETIME_COLUMN, kind="stable").reset_index(drop=True)

if df.empty:
    raise ValueError("No valid observations remain after preprocessing.")
if (df[TARGET_COLUMN] <= 0).any():
    raise ValueError("Target contains non-positive values; percentage errors are undefined.")

log_event(
    "Preprocessing completed: "
    f"removed {invalid_datetime_count} invalid datetimes and "
    f"{invalid_target_count} invalid target values; retained valid duplicates."
)


## 5 Time-Series Integrity Checks


In [ ]:
duplicate_timestamp_count = int(df[DATETIME_COLUMN].duplicated(keep=False).sum())
time_differences = df[DATETIME_COLUMN].diff()
non_hourly_mask = time_differences.notna() & (time_differences != pd.Timedelta(hours=1))
non_hourly_interval_count = int(non_hourly_mask.sum())

print(f"Rows participating in duplicate timestamps: {duplicate_timestamp_count:,}")
print(f"Non-hourly consecutive intervals: {non_hourly_interval_count:,}")
if non_hourly_interval_count:
    display(
        pd.DataFrame({
            "previous_datetime": df[DATETIME_COLUMN].shift(1)[non_hourly_mask],
            "current_datetime": df.loc[non_hourly_mask, DATETIME_COLUMN],
            "interval": time_differences[non_hourly_mask],
        }).head(20)
    )

DST_NOTE = (
    "Duplicate timestamps correspond to daylight-saving transitions and are expected. "
    "Non-hourly intervals are expected due to daylight-saving transitions. "
    "These are not preprocessing errors and were preserved."
)
print(DST_NOTE)
log_event(
    "DST checks completed: "
    f"{duplicate_timestamp_count} duplicate-timestamp rows and "
    f"{non_hourly_interval_count} non-hourly intervals documented and retained."
)


## 6 Exploratory Data Analysis


In [ ]:
# Complete demand series
plt.figure(figsize=(14, 5))
plt.plot(df[DATETIME_COLUMN], df[TARGET_COLUMN], linewidth=0.45, color="#1f77b4")
plt.title("PJM East Hourly Electricity Demand")
plt.xlabel("Datetime")
plt.ylabel("Demand (MW)")
save_figure("time_series.png")

# Demand distribution
plt.figure(figsize=(10, 5))
plt.hist(df[TARGET_COLUMN], bins=50, color="#4c78a8", edgecolor="white")
plt.title("Distribution of PJM East Electricity Demand")
plt.xlabel("Demand (MW)")
plt.ylabel("Frequency")
save_figure("load_distribution.png")

# Average demand by hour
hourly_average = df.groupby(df[DATETIME_COLUMN].dt.hour)[TARGET_COLUMN].mean()
plt.figure(figsize=(10, 5))
plt.plot(hourly_average.index, hourly_average.values, marker="o", color="#f58518")
plt.xticks(range(24))
plt.title("Average Electricity Demand by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Demand (MW)")
save_figure("average_demand_by_hour.png")

# Average demand by day of week
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily_average = df.groupby(df[DATETIME_COLUMN].dt.dayofweek)[TARGET_COLUMN].mean().reindex(range(7))
plt.figure(figsize=(10, 5))
plt.bar(day_names, daily_average.values, color="#54a24b")
plt.title("Average Electricity Demand by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Demand (MW)")
plt.xticks(rotation=25, ha="right")
save_figure("average_demand_by_day.png")
log_event("Exploratory analysis completed and four figures saved.")


## 7 Feature Engineering


In [ ]:
def create_features(frame, target_column, datetime_column, n_lags):
    """Create lag and calendar predictors without altering valid timestamps."""
    featured = frame.copy()
    for lag in range(1, n_lags + 1):
        featured[f"lag_{lag}"] = featured[target_column].shift(lag)

    timestamps = featured[datetime_column].dt
    featured["hour"] = timestamps.hour
    featured["dayofweek"] = timestamps.dayofweek
    featured["month"] = timestamps.month
    featured["dayofyear"] = timestamps.dayofyear
    featured["weekofyear"] = timestamps.isocalendar().week.astype("int16")
    featured["is_weekend"] = (timestamps.dayofweek >= 5).astype("int8")
    return featured.dropna().reset_index(drop=True)


model_df = create_features(df, TARGET_COLUMN, DATETIME_COLUMN, N_LAGS)
FEATURE_COLUMNS = [f"lag_{lag}" for lag in range(1, N_LAGS + 1)] + [
    "hour", "dayofweek", "month", "dayofyear", "weekofyear", "is_weekend"
]
PROCESSED_ROWS = len(model_df)

if model_df.empty:
    raise ValueError("Feature engineering produced an empty dataset.")
if model_df[FEATURE_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError("Missing values remain in the modeling matrix.")

print(f"Modeling rows: {PROCESSED_ROWS:,}")
print(f"Predictor count: {len(FEATURE_COLUMNS)}")
display(model_df.head())
log_event(f"Feature engineering completed with {len(FEATURE_COLUMNS)} predictors.")


## 8 Save Processed Dataset


In [ ]:
processed_data_path = PROCESSED_DIR / PROCESSED_FILENAME
model_df.to_csv(processed_data_path, index=False)
if not processed_data_path.is_file() or processed_data_path.stat().st_size == 0:
    raise IOError(f"Processed dataset was not saved correctly: {processed_data_path}")
log_event(f"Processed dataset saved to {processed_data_path}.")


## 9 Chronological Train/Test Split


In [ ]:
split_index = int(len(model_df) * TRAIN_FRACTION)
if split_index <= 0 or split_index >= len(model_df):
    raise ValueError("Chronological split requires non-empty training and testing sets.")

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print(f"Training samples: {len(train_df):,} ({len(train_df) / len(model_df):.1%})")
print(f"Testing samples:  {len(test_df):,} ({len(test_df) / len(model_df):.1%})")
print(f"Train period: {train_df[DATETIME_COLUMN].iloc[0]} to {train_df[DATETIME_COLUMN].iloc[-1]}")
print(f"Test period:  {test_df[DATETIME_COLUMN].iloc[0]} to {test_df[DATETIME_COLUMN].iloc[-1]}")

plt.figure(figsize=(14, 5))
plt.plot(train_df[DATETIME_COLUMN], y_train, label="Training set", linewidth=0.5)
plt.plot(test_df[DATETIME_COLUMN], y_test, label="Testing set", linewidth=0.5)
plt.axvline(test_df[DATETIME_COLUMN].iloc[0], color="black", linestyle="--", label="80/20 split")
plt.title("Chronological Train/Test Split")
plt.xlabel("Datetime")
plt.ylabel("Demand (MW)")
plt.legend()
save_figure("train_test_split.png")
log_event("Chronological 80/20 train/test split completed without shuffling.")


## 10 Evaluation Function


In [ ]:
def evaluate_predictions(model_name, actual, predicted, training_time_seconds):
    """Return the required metrics for a model's aligned predictions."""
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    if actual_array.shape != predicted_array.shape:
        raise ValueError(f"Shape mismatch for {model_name}: {actual_array.shape} vs {predicted_array.shape}")
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual_array, predicted_array),
        "RMSE": mean_squared_error(actual_array, predicted_array) ** 0.5,
        "MAPE": mean_absolute_percentage_error(actual_array, predicted_array) * 100,
        "R2": r2_score(actual_array, predicted_array),
        "Training_Time_Seconds": float(training_time_seconds),
    }


def plot_predictions(datetimes, actual, predicted, title, filename):
    """Plot the first configured week of test predictions."""
    plot_count = min(PREDICTION_PLOT_HOURS, len(actual))
    plt.figure(figsize=(14, 5))
    plt.plot(datetimes.iloc[:plot_count], np.asarray(actual)[:plot_count], label="Actual", linewidth=1.8)
    plt.plot(datetimes.iloc[:plot_count], np.asarray(predicted)[:plot_count], label="Predicted", linewidth=1.4)
    plt.title(title)
    plt.xlabel("Datetime")
    plt.ylabel("Demand (MW)")
    plt.legend()
    save_figure(filename)


metrics_records = []
prediction_store = {}
log_event("Shared evaluation and plotting functions initialized.")


## 11 Persistence Baseline


In [ ]:
# One-hour persistence uses the immediately preceding observed demand (lag_1).
start_time = time.perf_counter()
persistence_predictions = X_test["lag_1"].to_numpy()
persistence_training_time = time.perf_counter() - start_time

metrics_records.append(
    evaluate_predictions("Persistence", y_test, persistence_predictions, persistence_training_time)
)
prediction_store["Persistence_Prediction"] = persistence_predictions
plot_predictions(
    test_df[DATETIME_COLUMN], y_test, persistence_predictions,
    "Persistence Baseline: Actual vs Predicted (First Test Week)",
    "persistence_prediction.png",
)
display(pd.DataFrame([metrics_records[-1]]).round(4))
log_event("Persistence baseline evaluated.")


## 12 Linear Regression


In [ ]:
linear_regression = LinearRegression()
start_time = time.perf_counter()
linear_regression.fit(X_train, y_train)
linear_training_time = time.perf_counter() - start_time
linear_predictions = linear_regression.predict(X_test)

metrics_records.append(
    evaluate_predictions("Linear Regression", y_test, linear_predictions, linear_training_time)
)
prediction_store["Linear_Regression_Prediction"] = linear_predictions
save_model(linear_regression, MODELS_DIR / "linear_regression_model.pkl")
plot_predictions(
    test_df[DATETIME_COLUMN], y_test, linear_predictions,
    "Linear Regression: Actual vs Predicted (First Test Week)",
    "linear_regression_prediction.png",
)
display(pd.DataFrame([metrics_records[-1]]).round(4))
log_event(f"Linear regression trained, evaluated, and saved in {linear_training_time:.3f} seconds.")


## 13 Random Forest


In [ ]:
random_forest = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
start_time = time.perf_counter()
random_forest.fit(X_train, y_train)
rf_training_time = time.perf_counter() - start_time
rf_predictions = random_forest.predict(X_test)

save_model(random_forest, MODELS_DIR / "random_forest_model.pkl")
metrics_records.append(
    evaluate_predictions("Random Forest", y_test, rf_predictions, rf_training_time)
)
prediction_store["Random_Forest_Prediction"] = rf_predictions
plot_predictions(
    test_df[DATETIME_COLUMN], y_test, rf_predictions,
    "Random Forest: Actual vs Predicted (First Test Week)",
    "random_forest_prediction.png",
)
display(pd.DataFrame([metrics_records[-1]]).round(4))
log_event(f"Random forest trained, evaluated, and saved in {rf_training_time:.3f} seconds.")


## 14 Model Comparison


In [ ]:
results_df = pd.DataFrame(metrics_records)
results_df = results_df.sort_values("RMSE", ascending=True).reset_index(drop=True)
display(results_df.style.format({
    "MAE": "{:.2f}", "RMSE": "{:.2f}", "MAPE": "{:.3f}%",
    "R2": "{:.4f}", "Training_Time_Seconds": "{:.3f}",
}))

plt.figure(figsize=(10, 5))
bars = plt.bar(results_df["Model"], results_df["RMSE"], color=["#4c78a8", "#f58518", "#54a24b"])
plt.bar_label(bars, fmt="%.1f", padding=3)
plt.title("Model Comparison by Test RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE (MW; lower is better)")
save_figure("model_comparison_rmse.png")
log_event("All three models compared using the required metrics.")


## 15 Feature Importance


In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": FEATURE_COLUMNS,
    "Importance": random_forest.feature_importances_,
}).sort_values("Importance", ascending=False, ignore_index=True)

feature_importance_path = RESULTS_DIR / "random_forest_feature_importance.csv"
feature_importance_df.to_csv(feature_importance_path, index=False)
display(feature_importance_df.head(15))

plot_importance = feature_importance_df.head(15).sort_values("Importance")
plt.figure(figsize=(10, 7))
plt.barh(plot_importance["Feature"], plot_importance["Importance"], color="#4c78a8")
plt.title("Random Forest Feature Importance (Top 15)")
plt.xlabel("Impurity-Based Importance")
plt.ylabel("Feature")
save_figure("random_forest_feature_importance.png")
log_event("Random forest feature importance calculated and saved.")


## 16 Save Outputs


In [ ]:
benchmark_results_path = RESULTS_DIR / "benchmark_results.csv"
predictions_path = RESULTS_DIR / "predictions.csv"
metrics_path = RESULTS_DIR / "metrics.txt"

results_df.to_csv(benchmark_results_path, index=False)
predictions_df = pd.DataFrame({
    DATETIME_COLUMN: test_df[DATETIME_COLUMN].to_numpy(),
    "Actual": y_test.to_numpy(),
    **prediction_store,
})
predictions_df.to_csv(predictions_path, index=False)

metrics_lines = ["PJM Benchmark Metrics", "=" * 80]
for record in metrics_records:
    metrics_lines.extend([
        "",
        f"Model: {record['Model']}",
        f"MAE: {record['MAE']:.6f}",
        f"RMSE: {record['RMSE']:.6f}",
        f"MAPE: {record['MAPE']:.6f}%",
        f"R²: {record['R2']:.6f}",
        f"Training time: {record['Training_Time_Seconds']:.6f} seconds",
    ])
metrics_path.write_text("\n".join(metrics_lines) + "\n", encoding="utf-8")
log_event("Benchmark results, predictions, and human-readable metrics saved.")


## 17 Benchmark Summary


In [ ]:
best_row = results_df.iloc[0]
summary = {
    "Dataset": "PJM East (PJME) hourly electricity demand",
    "Original rows": ORIGINAL_ROWS,
    "Processed rows": PROCESSED_ROWS,
    "Number of features": len(FEATURE_COLUMNS),
    "Training samples": len(train_df),
    "Testing samples": len(test_df),
    "Best model": best_row["Model"],
    "Best MAE": best_row["MAE"],
    "Best RMSE": best_row["RMSE"],
    "Best MAPE": best_row["MAPE"],
    "Best R²": best_row["R2"],
    "Training time": best_row["Training_Time_Seconds"],
    "Python version": platform.python_version(),
    "DST note": DST_NOTE,
}

print("BENCHMARK SUMMARY")
print("=" * 80)
for key, value in summary.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")
log_event(f"Benchmark summary prepared; best model by RMSE: {best_row['Model']}.")


## 18 Final Validation


In [ ]:
TOTAL_RUNTIME = time.perf_counter() - BENCHMARK_START_TIME
summary["Total notebook runtime"] = TOTAL_RUNTIME

environment_path = RESULTS_DIR / "environment.txt"
environment_details = [
    f"Python version: {platform.python_version()}",
    f"Platform: {platform.platform()}",
    f"pandas version: {pd.__version__}",
    f"numpy version: {np.__version__}",
    f"matplotlib version: {matplotlib.__version__}",
    f"scikit-learn version: {sklearn.__version__}",
    f"joblib version: {joblib.__version__}",
    f"Working directory: {NOTEBOOK_DIR}",
    f"Benchmark runtime: {TOTAL_RUNTIME:.6f} seconds",
    f"Run started: {RUN_STARTED_AT.isoformat(timespec='seconds')}",
]
environment_path.write_text("\n".join(environment_details) + "\n", encoding="utf-8")

log_event(f"Total notebook runtime: {TOTAL_RUNTIME:.3f} seconds.")
log_event("Beginning final artifact validation.")
run_log_path = RESULTS_DIR / "run_log.txt"
run_log_path.write_text("\n".join(LOG_MESSAGES) + "\n", encoding="utf-8")

expected_figures = [
    "time_series.png",
    "load_distribution.png",
    "average_demand_by_hour.png",
    "average_demand_by_day.png",
    "train_test_split.png",
    "persistence_prediction.png",
    "linear_regression_prediction.png",
    "random_forest_prediction.png",
    "model_comparison_rmse.png",
    "random_forest_feature_importance.png",
]
expected_prediction_columns = [
    DATETIME_COLUMN,
    "Actual",
    "Persistence_Prediction",
    "Linear_Regression_Prediction",
    "Random_Forest_Prediction",
]

validation_errors = []

required_files = {
    "processed dataset": processed_data_path,
    "benchmark results": benchmark_results_path,
    "predictions": predictions_path,
    "metrics": metrics_path,
    "feature importance": feature_importance_path,
    "environment": environment_path,
    "run log": run_log_path,
    "linear regression model": MODELS_DIR / "linear_regression_model.pkl",
    "random forest model": MODELS_DIR / "random_forest_model.pkl",
}
for label, path in required_files.items():
    if not path.is_file():
        validation_errors.append(f"Missing {label}: {path}")
    elif path.stat().st_size == 0:
        validation_errors.append(f"Empty {label}: {path}")

for filename in expected_figures:
    figure_path = FIGURES_DIR / filename
    if not figure_path.is_file() or figure_path.stat().st_size == 0:
        validation_errors.append(f"Missing or empty figure: {figure_path}")

if processed_data_path.is_file() and pd.read_csv(processed_data_path).empty:
    validation_errors.append("Processed dataset exists but contains no rows.")
if benchmark_results_path.is_file() and len(pd.read_csv(benchmark_results_path)) != 3:
    validation_errors.append("benchmark_results.csv must contain exactly three model rows.")
if predictions_path.is_file():
    saved_prediction_columns = pd.read_csv(predictions_path, nrows=0).columns.tolist()
    if saved_prediction_columns != expected_prediction_columns:
        validation_errors.append(
            "predictions.csv columns are incorrect. "
            f"Expected {expected_prediction_columns}; found {saved_prediction_columns}."
        )
if feature_importance_path.is_file() and pd.read_csv(feature_importance_path).empty:
    validation_errors.append("Feature importance file exists but is empty.")

if validation_errors:
    raise RuntimeError("Final validation failed:\n- " + "\n- ".join(validation_errors))

log_event("Final validation passed: every required artifact is present, non-empty, and structurally valid.")
run_log_path.write_text("\n".join(LOG_MESSAGES) + "\n", encoding="utf-8")
if not run_log_path.is_file() or run_log_path.stat().st_size == 0:
    raise IOError(f"Run log was not saved correctly: {run_log_path}")

print("\nFINAL VALIDATION PASSED")
print("\nBENCHMARK SUMMARY (FINAL)")
print("=" * 80)
for key, value in summary.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")
print(f"Total notebook runtime: {TOTAL_RUNTIME:.3f} seconds")
print(f"Artifacts saved under: {PROJECT_ROOT}")


## 19 Conclusions

The benchmark is complete when the final validation cell reports success. Model rankings should be interpreted on the untouched chronological test period; lower MAE, RMSE, and MAPE and higher R² indicate better predictive performance. The persistence model provides the essential short-horizon baseline, while the two trained estimators quantify the benefit of adding calendar and lag information.

PJM daylight-saving duplicate timestamps and non-hourly intervals are documented and preserved by design. This ensures that benchmark preprocessing does not silently alter valid source observations.
